In [ ]:
import pyfinancialdata
import pandas as pd
import numpy as np
from scipy import stats
import os
import glob

def fetch_oanda_data(instrument_name, year_list):
    # Locate the data directory inside the installed package
    pkg_dir = os.path.dirname(pyfinancialdata.__file__)
    
    # Check common installation paths for the 'data' folder
    data_dir = os.path.join(pkg_dir, 'data')
    if not os.path.exists(data_dir):
        # If installed via pip archive, data might sit one level up in site-packages
        data_dir = os.path.join(os.path.dirname(pkg_dir), 'data')

    yearly_data = []
    for year in year_list:
        # Construct the exact path pattern the package uses internally
        path_pattern = os.path.join(data_dir, 'currencies', 'oanda', instrument_name, str(year), '*.csv')
        csv_files = glob.glob(path_pattern)
        
        if not csv_files:
            print(f"Warning: No files found for {year} at {path_pattern}")
            continue
            
        # Read and combine all CSVs for the year using modern pd.concat
        dfs = [pd.read_csv(f, index_col=0, parse_dates=True) for f in csv_files]
        yearly_data.append(pd.concat(dfs))
        
    if not yearly_data:
        raise FileNotFoundError("Could not locate the dataset CSVs in the installation directory.")
        
    # Combine all requested years and sort chronologically
    df_combined = pd.concat(yearly_data).sort_index()
    return df_combined

def build_intraday_series(instrument_name, year_list):
    # 1. Fetch data using the custom loader
    df = fetch_oanda_data(instrument_name, year_list)
    
    # 2. Resample to 5-minute bars and calculate log returns
    df_5m = df['close'].resample('5min').last().dropna().to_frame()
    df_5m['log_return'] = np.log(df_5m['close'] / df_5m['close'].shift(1))
    #df_5m['log_return'] = (np.log(df_5m['close'] / df_5m['close'].shift(1))) ** 2
    df_5m = df_5m.dropna()
    
    # 3. Isolate the date for daily grouping
    df_5m['day'] = df_5m.index.date
    daily_counts = df_5m.groupby('day').size()
    
    # 4. Apply the 50% modal count filter
    modal_count = stats.mode(daily_counts, keepdims=True).mode[0]
    valid_days = daily_counts[daily_counts >= 0.5 * modal_count].index
    
    df_filtered = df_5m[df_5m['day'].isin(valid_days)].copy()
    df_filtered['asset_name'] = instrument_name
    
    return df_filtered.reset_index()

# Extract EUR_USD data for specific years
df_eurusd = build_intraday_series(
    instrument_name='EUR_USD', #EUR_USD
    year_list=[2018]
)

df_eurusd.to_csv('intraday_returns_raw.csv', index=False)